# Cross-Arm Significance Testing

This notebook is the "Comparison across arms" step (proposal, Table 1), separate from Arm A (`global_model_armA.ipynb`), Arm B (`clinical_stratified_armB.ipynb`), and Arm C (`data_driven_armC.ipynb`). It reads only the fold-level result CSVs each arm already saved and does not fit, tune, or predict anything itself.

- **RQ1** (Arm A vs Arm B): does clinical (sex) stratification change performance relative to the non-stratified baseline?
- **RQ2** (Arm A vs Arm C): does data-driven clustering change performance relative to the non-stratified baseline?
- **RQ3** (Arm B vs Arm C): how do the two stratification strategies compare to each other?

Every arm reports mean ± standard deviation over 5 outer folds and stops there. That is not the same as knowing whether an observed gap exceeds fold-to-fold noise: with 5 folds of roughly 60 patients each, three patients changing class moves accuracy by about 5 points. This notebook adds paired significance testing across the shared folds, plus a plain sign-count, to put a number on how much of each arm-to-arm gap is distinguishable from noise at this sample size -- and, per the caveat in Section 5, how much confidence even that number deserves.

## 2. Load fold-level results and verify comparability

All three arms are loaded and checked to cover the same 5 outer folds and the same two model names before anything is compared -- if any arm's fold-level file diverges (different fold count, different model names), the comparisons below would silently be comparing different things.

In [1]:
import os

import numpy as np
import pandas as pd
from scipy import stats

# Path constants -- this notebook lives in notebooks/, so PROJECT_ROOT is
# one level up; it reads only fold-level results already saved by the
# three arm notebooks, and writes its own summary back into results/.
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

fold_files = {
    "A": os.path.join(RESULTS_DIR, "armA_fold_results.csv"),
    "B": os.path.join(RESULTS_DIR, "armB_fold_results.csv"),
    "C": os.path.join(RESULTS_DIR, "armC_fold_results.csv"),
}
metrics = ["accuracy", "f1", "roc_auc"]

fold_data = {arm: pd.read_csv(path) for arm, path in fold_files.items()}

folds_by_arm = {arm: sorted(df["fold"].unique().tolist()) for arm, df in fold_data.items()}
models_by_arm = {arm: sorted(df["model"].unique().tolist()) for arm, df in fold_data.items()}

assert len(set(map(tuple, folds_by_arm.values()))) == 1, f"Fold sets differ across arms: {folds_by_arm}"
assert len(set(map(tuple, models_by_arm.values()))) == 1, f"Model name sets differ across arms: {models_by_arm}"

FOLDS = folds_by_arm["A"]
MODELS = models_by_arm["A"]
print(f"All three arms cover the same {len(FOLDS)} folds {FOLDS} and the same models {MODELS}.")

All three arms cover the same 5 folds [0, 1, 2, 3, 4] and the same models ['logreg', 'rf'].


## 3. Paired comparisons per classifier and metric

For each classifier and each metric, the three arms' fold-level values are paired by fold (fold 0's Arm A accuracy against fold 0's Arm B accuracy, and so on) -- valid because all three arms share the identical outer fold partition (`fold_id.csv`), so "fold 2" means the same 59-60 patients in every arm. For each of the three pairwise comparisons (A-vs-B for RQ1, A-vs-C for RQ2, B-vs-C for RQ3):

- **Mean paired difference** -- the average, across the 5 folds, of (first arm's value minus second arm's value).
- **Paired t-test** (`scipy.stats.ttest_rel`) -- tests whether that mean difference is distinguishable from zero, assuming the fold-level differences are approximately normal.
- **Wilcoxon signed-rank test** (`scipy.stats.wilcoxon`) -- a non-parametric alternative that does not assume normality, included because n=5 is too small to check the t-test's assumption with any confidence.
- **Sign count** -- how many of the 5 folds each arm wins on that metric, with ties reported separately. This is reported because a p-value at n=5 can look more precise than it is; "won 4 of 5 folds" is an honest, easily-checked alternative summary of the same information.

In [2]:
def paired_stats(df_x, df_y, model, metric):
    x = df_x[df_x["model"] == model].sort_values("fold")[metric].to_numpy()
    y = df_y[df_y["model"] == model].sort_values("fold")[metric].to_numpy()
    diff = x - y

    t_stat, t_p = stats.ttest_rel(x, y)
    try:
        w_stat, w_p = stats.wilcoxon(x, y)
    except ValueError:
        # wilcoxon raises if all differences are exactly zero
        w_stat, w_p = np.nan, np.nan

    wins_x = int((diff > 0).sum())
    wins_y = int((diff < 0).sum())
    ties = int((diff == 0).sum())

    return {
        "mean_diff": diff.mean(),
        "t_stat": t_stat, "t_p": t_p,
        "wilcoxon_stat": w_stat, "wilcoxon_p": w_p,
        "wins_x": wins_x, "wins_y": wins_y, "ties": ties,
    }

comparisons = [("A", "B", "RQ1"), ("A", "C", "RQ2"), ("B", "C", "RQ3")]

rows = []
for arm_x, arm_y, rq in comparisons:
    for model in MODELS:
        for metric in metrics:
            stats_row = paired_stats(fold_data[arm_x], fold_data[arm_y], model, metric)
            rows.append({
                "rq": rq, "comparison": f"{arm_x} vs {arm_y}", "model": model, "metric": metric,
                **stats_row,
            })

cross_arm_df = pd.DataFrame(rows)
cross_arm_df.to_csv(os.path.join(RESULTS_DIR, "cross_arm_comparison.csv"), index=False)
print("Saved cross_arm_comparison.csv")
cross_arm_df.round(3)

Saved cross_arm_comparison.csv


,rq,comparison,model,metric,mean_diff,t_stat,t_p,wilcoxon_stat,wilcoxon_p,wins_x,wins_y,ties
0,RQ1,A vs B,logreg,accuracy,0.057,3.310,0.030,0.0,0.125,4,0,1
1,RQ1,A vs B,logreg,f1,0.067,3.122,0.035,0.0,0.125,4,0,1
2,RQ1,A vs B,logreg,roc_auc,0.013,0.888,0.425,4.0,0.438,3,2,0
3,RQ1,A vs B,rf,accuracy,0.013,1.078,0.342,5.0,0.562,4,1,0
4,RQ1,A vs B,rf,f1,0.006,0.377,0.726,6.0,0.812,3,2,0
5,RQ1,A vs B,rf,roc_auc,-0.003,-0.268,0.802,7.0,1.000,3,2,0
6,RQ2,A vs C,logreg,accuracy,0.027,1.170,0.307,2.0,0.375,3,1,1
7,RQ2,A vs C,logreg,f1,0.028,1.511,0.205,3.0,0.312,3,2,0
8,RQ2,A vs C,logreg,roc_auc,0.022,1.149,0.315,4.0,0.438,3,2,0
9,RQ2,A vs C,rf,accuracy,0.013,0.866,0.435,6.0,0.812,3,2,0


## 4. Summary mapped onto RQ1 / RQ2 / RQ3

The accuracy row for each comparison is pulled out below as the headline figure (F1 and ROC-AUC are in the full table above and in `cross_arm_comparison.csv`), since accuracy is the metric the proposal names as the one enabling comparison with prior heart-disease studies (Section 3.5).

In [3]:
headline = (
    cross_arm_df[cross_arm_df["metric"] == "accuracy"]
    [["rq", "comparison", "model", "mean_diff", "t_p", "wilcoxon_p", "wins_x", "wins_y", "ties"]]
    .sort_values(["rq", "model"])
    .reset_index(drop=True)
)
headline

,rq,comparison,model,mean_diff,t_p,wilcoxon_p,wins_x,wins_y,ties
0,RQ1,A vs B,logreg,0.057119,0.029646,0.1250,4,0,1
1,RQ1,A vs B,rf,0.013333,0.341519,0.5625,4,1,0
2,RQ2,A vs C,logreg,0.027062,0.306807,0.3750,3,1,1
3,RQ2,A vs C,rf,0.013277,0.435422,0.8125,3,2,0
4,RQ3,B vs C,logreg,-0.030056,0.208008,0.2500,1,3,1
5,RQ3,B vs C,rf,-0.000056,0.996435,1.0000,2,2,1


## 5. Caveat: paired tests across CV folds are anti-conservative

**Paired t-tests (and Wilcoxon tests) across cross-validation folds systematically understate the true p-value.** The standard derivation of these tests assumes the paired observations are independent; outer CV folds are not independent, because their training sets overlap heavily -- with 5-fold CV, any two folds' training sets share 3 of the underlying 4 non-held-out fifths of the data. Models trained on substantially overlapping data produce correlated errors, which inflates apparent significance relative to a true independent-samples test. This is a well-known limitation of significance testing on cross-validated results (sometimes called the "Dietterich problem" after the 1998 analysis that first quantified it for paired resampling tests), not specific to this study or this dataset.

**No correction is applied here.** Proper corrections (e.g. a corrected resampled t-test, or nested CV specifically designed for inference) exist but add complexity disproportionate to what 5 folds can support either way. The practical reading of the p-values in Sections 3-4 is: treat every p-value here as an upper bound on the true significance, not a calibrated one -- a result reported as "p=0.03" should be read as "at least this uncertain, plausibly more," and even that one nominally significant comparison should be treated as weak evidence, not confirmation.